#### **Python memory management**

Memory management in python involves a combination of automatic garbage collection, reference counting, and various internal optimizations to efficiently management memory allocation and deallocation. Understanding these mechanisms can help developers write more efficient and robust applications.

1. key concepts in python memory management 
2. memory allocation and deallocation
3. reference counting
4. garbage collection
5. the gc module 
6. memory management best practices

---
**Reference counting**
    
+ It is the primary method python uses to manage memory. Each object in python maintains a count of references pointing to it. When the reference count drops to zero, the memory occupied by the object is deallocated.

In [1]:
import sys

a = []
print(sys.getrefcount(a))

2


why output as 2:
  + it is because 1 reference for list 'a'
  + another for 1 for getrefcount() method

In [2]:
b = a
print(sys.getrefcount(b))

3


In [3]:
print(f"num of ref. in a: {sys.getrefcount(a)}")
print(f"num of ref. in b: {sys.getrefcount(b)}")

del b
print(f"num of ref. in a: {sys.getrefcount(a)}")

# we can't access b cause b already deleted
# print(f"num of ref. in b: {sys.getrefcount(b)}")


num of ref. in a: 3
num of ref. in b: 3
num of ref. in a: 2


---
**garbage collection**
    
+ python includes a cyclic garbage collector to handle reference cycles. reference cycles occur when objects reference each other, preventing their reference counts from reaching zero.

In [4]:
import gc

In [5]:
# enable garbage collection
gc.enable()

In [6]:
# disable garbage collection
gc.disable()

In [7]:
# manually trigger garbage collection
gc.collect()
# the number of unreachable objects is returned

3

get_stats: returns a list of dictionaries containing per-generation statics

In [8]:
# get garbage collection stats
print(gc.get_stats())

print()
for stat in gc.get_stats():
    print(stat)

[{'collections': 75, 'collected': 1506, 'uncollectable': 0}, {'collections': 6, 'collected': 72, 'uncollectable': 0}, {'collections': 1, 'collected': 3, 'uncollectable': 0}]

{'collections': 75, 'collected': 1506, 'uncollectable': 0}
{'collections': 6, 'collected': 72, 'uncollectable': 0}
{'collections': 1, 'collected': 3, 'uncollectable': 0}


In [9]:
# get unreachable objects
print(gc.garbage)

[]


---
**Memory management best practices**

1. use local variables: local variables have a shorter life span and are freed sooner than global variables.
2. avoid circular references: circular references can lead to memory leaks if not properly managed. 
3. use generators: generators produce items one at a time and only keep one item in memory at a time, making them memory efficient.
4. explicitly delete objects: use the del statement to delete variables and objects explicitly.
5. profile memory usage: use memory profiling tools like tracemalloc and memory_profiler to identify memory leaks and optimize memory usage.

In [10]:
import gc

In [11]:
class MyObj:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")
    
    def __del__(self):
        print(f"object {self.name} deleted")
    

In [12]:
# creating object
obj1 = MyObj('object1')
obj2 = MyObj('object2')

# create circular reference
obj1.ref = obj2
obj2.ref = obj1

# deleting
del obj1
del obj2

Object object1 created
Object object2 created


+ handling circular reference

In [13]:
# creating object
obj1 = MyObj('object1')
obj2 = MyObj('object2')

# create circular reference
obj1.ref = obj2
obj2.ref = obj1

# deleting
del obj1
del obj2

# manually trigger the garbage collection
gc.collect()

Object object1 created
Object object2 created
object object1 deleted
object object2 deleted
object object1 deleted
object object2 deleted


4

---
**Generator for memory efficiency**

+ it allows you to produce items one at a time, using memory efficiently by only keeping one item in memory at a time.

In [14]:
def generate_numbers(n):
    for i in range(n):
        yield i

# using the generator
for num in generate_numbers(10000):
    print(num, end=" ")

    if num>20:
        break


0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 

---
**Profiling memory usage with tracemalloc**

In [15]:
import tracemalloc 

def create_list():
    return [i for i in range(10000)]

def main():
    tracemalloc.start()
    
    create_list()

    snapshot = tracemalloc.take_snapshot()
    # lineno: mean line number
    top_stats = snapshot.statistics('lineno')

    print("[ Top 10 ]")
    for stat in top_stats[:10]:
        print(stat)

In [16]:
main()

[ Top 10 ]
C:\Users\siyam\AppData\Local\Temp\ipykernel_4900\643711872.py:4: size=3200 B, count=100, average=32 B


In [ ]:
import tracemalloc 

def create_list():
    return [i for i in range(10000)]

def main():
    tracemalloc.start()
    
    create_list()

    snapshot = tracemalloc.take_snapshot()
    # lineno: mean line number
    top_stats = snapshot.statistics('lineno')

    print("[ Top 10 ]")
    for stat in top_stats[::]:
        print(stat)

main()

[ Top 10 ]
D:\Program Files\Python314\Lib\inspect.py:2179: size=50.8 KiB, count=2, average=25.4 KiB
d:\Rat_Race\AI-ML\env\Lib\site-packages\IPython\core\compilerop.py:86: size=15.9 KiB, count=179, average=91 B
d:\Rat_Race\AI-ML\env\Lib\site-packages\IPython\core\compilerop.py:174: size=13.2 KiB, count=136, average=99 B
d:\Rat_Race\AI-ML\env\Lib\site-packages\zmq\sugar\socket.py:806: size=5564 B, count=107, average=52 B
d:\Rat_Race\AI-ML\env\Lib\site-packages\IPython\core\inputtransformer2.py:101: size=4827 B, count=102, average=47 B
D:\Program Files\Python314\Lib\json\decoder.py:361: size=4510 B, count=44, average=102 B
d:\Rat_Race\AI-ML\env\Lib\site-packages\zmq\sugar\attrsettr.py:45: size=4416 B, count=94, average=47 B
D:\Program Files\Python314\Lib\codeop.py:117: size=3632 B, count=48, average=76 B
C:\Users\siyam\AppData\Local\Temp\ipykernel_4900\3113040107.py:4: size=3256 B, count=101, average=32 B
d:\Rat_Race\AI-ML\env\Lib\site-packages\jupyter_client\jsonutil.py:112: size=2650 B,